In [1]:
import reconstructions
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import torchvision

In [2]:
bs = 64  # batch size
depth = 50  # number of hidden layers
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")  # check for devices
print(f"Device: {device} with {torch.cuda.device_count()} gpus")

# load training/test data
trans = torchvision.transforms.Compose([torchvision.transforms.ToTensor()])
train_data = torchvision.datasets.CIFAR10("./", train=True, download=True, transform=trans)
val_data = torchvision.datasets.CIFAR10("./", train=False, transform=trans, download=True)
train_data_loader = torch.utils.data.DataLoader(train_data, batch_size=bs, shuffle=True)
val_data_loader = torch.utils.data.DataLoader(val_data, batch_size=bs, shuffle=False)

Device: cpu with 0 gpus


In [3]:
# dataset parameters
input_size = 32
output_size = 10

# network parameters
latent_space_size = 64
heads = 4
mlp_dim = 256
depth = 2
patch_size = 4
channels = 3

# calculate intermediate sizes
num_patches = (input_size // patch_size) ** 2
patch_dim = channels * patch_size * patch_size

# setup layer structure
layers = [
        reconstructions.Rearrange("b c (h p1) (w p2) -> b (h w) (p1 p2 c)", p1=patch_size, p2=patch_size),
        nn.Linear(patch_dim, latent_space_size),
        reconstructions.PosEncoding(num_patches, latent_space_size),
    ]

# add attention blocks
for _ in range(depth):
        layers += [
            reconstructions.Parallel(
                nn.Sequential(
                    nn.LayerNorm(latent_space_size),
                    reconstructions.Attention(latent_space_size, heads),
                ),
                nn.Sequential(nn.Identity()),  # skip connection
            ),
            reconstructions.Parallel(
                nn.Sequential(
                    nn.LayerNorm(latent_space_size),
                    nn.Linear(latent_space_size, mlp_dim),
                    nn.GELU(),
                    nn.Linear(mlp_dim, latent_space_size),
                ),
                nn.Sequential(nn.Identity()),  # skip connection
            ),
        ]


# add final layer blocks
layers += [
        reconstructions.StripActivation(),
        nn.Linear(latent_space_size, mlp_dim),
        nn.GELU(),
        nn.Linear(mlp_dim, output_size),
    ]


vit = nn.Sequential(*layers)

In [4]:
model_layers, co_model_layers = reconstructions.get_conet_layout(vit, batch=(next(iter(train_data))[0]).unsqueeze(0), device=device)
for l, c in zip(model_layers, co_model_layers):
    print(l)
    print(c)
    print("--------------------------------")

Sequential(
  (0): Rearrange(b c (h p1) (w p2) -> b (h w) (p1 p2 c), p1=4, p2=4, h=8)
)
Sequential(
  (0): Rearrange(b (h w) (p1 p2 c) -> b c (h p1) (w p2), p1=4, p2=4, h=8)
  (1): ReLU()
)
--------------------------------
Sequential(
  (0): Linear(in_features=48, out_features=64, bias=True)
)
Sequential(
  (0): Linear(in_features=64, out_features=48, bias=True)
)
--------------------------------
Sequential(
  (0): PosEncoding()
)
Sequential(
  (0): RePosEncoding(
    (pem): PosEncoding()
  )
)
--------------------------------
Sequential(
  (0): Parallel(
    (0): Sequential(
      (0): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
      (1): Attention(
        (to_qkv): Linear(in_features=64, out_features=192, bias=False)
        (to_out): Linear(in_features=64, out_features=64, bias=True)
      )
    )
    (1): Sequential(
      (0): Identity()
    )
  )
)
Sequential(
  (0): Parallel(
    (0): Sequential(
      (0): Sequential(
        (0): ReAttention(
          (a

In [ ]:
co_model = reconstructions.ContraNetwork(torch.nn.Sequential(*model_layers), torch.nn.Sequential(*co_model_layers),
                                         device=device)
# with steps we can limit the number of updates. For MLPs around 200 is sufficient
co_model.train(train_data_loader, its=1)


Epoch: 0 / 1
